# 🧪 Molecules as Text: SMILES & Cheminformatics with RDKit
### AI Literacy Project Taster Day — Queen Mary University of London

---

Welcome! In this notebook you will discover how chemists **write molecules as text strings** so that computers can read, draw, and analyse them — the same technology used in pharmaceutical companies to screen millions of drug candidates.

By the end of this tutorial you will be able to:
- Explain the rules for writing **SMILES** (Simplified Molecular-Input Line-Entry System) strings
- Use the **RDKit** Python library to create molecule objects from SMILES and draw them
- Calculate key **molecular properties** (molecular weight, number of rings, drug-likeness criteria)
- Measure **molecular similarity** between two drug compounds

> 💡 **How to run a cell:** Click on it, then press **Shift + Enter**. Run cells **in order from top to bottom**.

---


## Section 0 — Installing Required Packages 🛠️

Before we can work with molecules in Python, we need two specialist libraries:

| Library | What it does |
|---------|-------------|
| **RDKit** | The most widely used open-source cheminformatics toolkit — reads, draws, and analyses molecules |
| **DeepChem** | A deep learning library for chemistry and drug discovery (we'll use it to load a dataset of FDA-approved drugs) |

Run the cell below to install them. This may take a minute — you'll see a lot of text scroll past, which is normal.


In [ ]:
# install deepchem and rdkit
! pip install deepchem
! pip install rdkit

---
## Section 1 — What is SMILES? 🔤

### The problem: how do you store a molecule in a spreadsheet?

Chemists draw molecules as 2D structural diagrams — but a computer needs text or numbers to store and search them.

**SMILES** (Simplified Molecular-Input Line-Entry System) solves this by encoding a molecule as a short string of letters and symbols. For example:

| Molecule | SMILES |
|----------|--------|
| Water ($H_2O$) | `O` |
| Ethanol ($C_2H_5OH$) | `CCO` |
| Aspirin | `CC(=O)Oc1ccccc1C(=O)O` |
| Caffeine | `Cn1cnc2c1c(=O)n(c(=O)n2C)C` |

> 💬 Notice how even complex molecules like aspirin and caffeine can be described in a compact string!

---
### The 5 rules of SMILES

The rules below let you decode — or write — any SMILES string.


<div class="alert alert-success">
SMILES basic rules:

1. Atoms are labelled using their atomic symbols enclosed in square brackets, but if the atom belongs to the 'organic subset' (B, C, N, O, P, S, F, Cl, Br, and I), no brackets are needed (unless we want to specify for example the formal charge of the atom). Hydrogens bonded to an atom from the organic subset are normally implied (so for example the SMILES string for methane ($CH_{4}$) is just C as hydrogens are automatically added to fill in the free valencies).

2. Single and aromatic bonds are normally omitted. Double and triple bonds are represented with the "=" and "#" symbols, respectively. Ionic bonds are not explicitly represented.

3. Branches in the chemical structure are eclosed in round brackets:

<img src="branch1.png" alt="Drawing" style="width: 400px"/>

Branches can be nested.

4. Rings are represented by breaking one of the (single or aromatic) bonds in the ring and labelling the two atoms involved in the broken bond with the same number (see examples below).

5. Individual parts in disconnected compounds (e.g. ion pairs) are separated by a period.

</div>

### Visual examples

The images below show how the backbone (main chain) and branches map onto SMILES strings, and how ring closure numbers work.


In the examples below, the atoms not enclosed in round brackets represent the main 'backbone' of the molecule, while atoms enclosed in brackets represent the 'branches'.

![rings](rings.png)

> 💬 **Quick check:** Can you identify the backbone and branches in ibuprofen's SMILES `CC(C)Cc1ccc(cc1)C(C)C(=O)O`?  
> Count how many branch points (round brackets) there are.

---


## Section 2 — Loading Our Tools 📦

We now import all the RDKit modules we'll need — do this **once** at the start of any notebook.

Think of an import as "opening an app" before you use it.


In [ ]:
# ── Core RDKit modules ─────────────────────────────────────────────────────────
from rdkit.Chem import AllChem as Chem      # reads and manipulates molecules
from rdkit.Chem import Draw                 # draws molecules as images
from rdkit.Chem import rdMolDescriptors     # calculates molecular descriptors
from rdkit import DataStructs               # handles molecular fingerprints

# ── Data handling ──────────────────────────────────────────────────────────────
import pandas as pd                         # tables of data (like Excel)

print("✅ All tools loaded successfully!")


---
## Section 3 — SMILES in Action: Drawing Molecules 🎨

The key RDKit function is `Chem.MolFromSmiles()` — it takes a SMILES string and returns a **molecule object** that RDKit can work with.

### 3a — Your first molecule: acetic acid (vinegar!)

Acetic acid ($CH_3COOH$) has the SMILES `CC(=O)O`:
- First `C` = methyl carbon ($CH_3$)
- `C(=O)` = carbonyl carbon with a double bond to oxygen
- Final `O` = hydroxyl oxygen ($OH$)


In [ ]:
mol = Chem.MolFromSmiles('CC(=O)O')

> 💬 Does the structure look like acetic acid to you? Notice how RDKit automatically adds the implicit hydrogens when drawing.

---
### 3b — Try some molecules you know

Let's draw a few familiar molecules. Notice how each SMILES encodes the structure:


> 💬 **Challenge:** Can you work out which atoms correspond to which letters in the caffeine SMILES `Cn1cnc2c1c(=O)n(c(=O)n2C)C`?  
> Hint: lowercase letters like `n` and `c` represent aromatic atoms.

---
### 3c — Molecules with rings

Rings are encoded by **breaking one bond** and labelling both atoms with the same number.  
For example, cyclohexane is `C1CCCCC1` — atom 1 and the final atom are bonded to close the ring.


### 4. Canonical SMILES
It is important to note that the same chemical structure can be represented with different SMILES. For example, in the image below both (a) and (b) are valid representations of the same molecule:

![rings2](rings2.png)

Let's verify this by trying both (a) and (b):

> 💬 **What does this tell us?** The same molecule can be written in many different valid SMILES — this is important when building databases, which is why **canonical SMILES** exist (see Section 4).

---
### 3d — Stereochemistry: cis and trans isomers

SMILES can encode **stereochemistry** — the 3D arrangement of atoms around a double bond.  
`/` and `\` symbols indicate whether substituents are on the same side (*cis*, Z) or opposite sides (*trans*, E):


In [ ]:
mol = Chem.MolFromSmiles("Br/C=C\\F")
Draw.MolToImage(mol, size = (200, 200))

In [ ]:
mol = Chem.MolFromSmiles("Br/C=C/F")
Draw.MolToImage(mol, size = (200, 200))

In [ ]:
mol = Chem.MolFromSmiles("Br/C=C\\F")
Draw.MolToImage(mol, size = (200, 200))

> 💬 **Why does this matter in medicine?** Thalidomide is a famous example: one stereoisomer treated morning sickness, the other caused birth defects. SMILES encodes this distinction precisely.

> 🐍 **Python note:** You may have noticed `\\` in one SMILES. In Python strings, `\` is a special "escape character", so we need `\\` to represent a literal backslash.

---


## Section 4 — Canonical SMILES: One Molecule, One Unique String 🔑

The same molecule can be written as many different valid SMILES. For databases and search engines, we need a **unique, standardised** representation — called the **canonical SMILES**.

RDKit can generate the canonical SMILES for any molecule using `Chem.MolToSmiles()`.


> 💬 **Why is this useful?** If two researchers independently draw aspirin using different SMILES, the canonical form lets a computer confirm it's the same compound — essential for drug databases like ChEMBL and PubChem.

---


## Section 5 — Calculating Molecular Properties ⚗️

One of the most powerful features of RDKit is the ability to calculate **molecular descriptors** — numerical properties that describe a molecule's size, shape, and chemistry.

These descriptors are used in **drug discovery** to filter out compounds that are unlikely to work as drugs before spending money on lab experiments.

### Lipinski's Rule of Five 💊

In 1997, Christopher Lipinski analysed thousands of oral drugs and found that nearly all of them satisfy these four criteria (the "Rule of Five"):

| Property | Rule | Why it matters |
|----------|------|---------------|
| **Molecular weight** | ≤ 500 Da | Large molecules can't cross cell membranes |
| **LogP** (lipophilicity) | ≤ 5 | Too oily → poor water solubility |
| **H-bond donors** | ≤ 5 | Too many → can't cross lipid membranes |
| **H-bond acceptors** | ≤ 10 | Too many → can't cross lipid membranes |

Let's calculate these for a molecule. We'll use **ibuprofen** as our example:


In [ ]:
mol = Chem.MolFromSmiles("CC(C)Cc1ccc(cc1)C(C)C(=O)O")
rdMolDescriptors.CalcExactMolWt(mol)

> 💬 Is ibuprofen's molecular weight within Lipinski's limit of 500 Da?


> 💬 Does ibuprofen satisfy all four Lipinski criteria? Check each one!
>
> Module containing functions to compute molecular descriptors can be found here: https://www.rdkit.org/docs/source/rdkit.Chem.rdMolDescriptors.html


In [ ]:
# ── Write your code here ───────────


In [ ]:
rdMolDescriptors.CalcTPSA(mol)

> 💬 **What is TPSA?** Topological Polar Surface Area — the total surface area of polar atoms (N and O). Oral drugs typically have TPSA < 140 Å². Does ibuprofen qualify?

---
### Checking a whole set of molecules at once

Now let's apply these descriptors to a **real dataset of FDA-approved drugs** from the DeepChem library.


Now let's compute all four Lipinski descriptors for every drug in the dataset using a **for loop**:

In [ ]:
# ── Calculate Lipinski descriptors for every drug in the FDA dataset ───────────
MolWt_list  = []   # molecular weight
HBD_list    = []   # hydrogen bond donors
HBA_list    = []   # hydrogen bond acceptors
TPSA_list   = []   # topological polar surface area

FDA = pd.read_csv('FDA_smiles.csv', index_col = 0)

for smiles in FDA['SMILES']:
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:                                      # skip invalid SMILES
        MolWt_list.append(rdMolDescriptors.CalcExactMolWt(mol))
        HBD_list.append(rdMolDescriptors.CalcNumHBD(mol))
        HBA_list.append(rdMolDescriptors.CalcNumHBA(mol))
        TPSA_list.append(rdMolDescriptors.CalcTPSA(mol))
    else:
        MolWt_list.append(None)
        HBD_list.append(None)
        HBA_list.append(None)
        TPSA_list.append(None)

# ── Add the results as new columns in the FDA DataFrame ───────────────────────
FDA['MolWt'] = MolWt_list
FDA['HBD']   = HBD_list
FDA['HBA']   = HBA_list
FDA['TPSA']  = TPSA_list

print(f"✅ Descriptors calculated for {len(FDA)} compounds.")
FDA.head()

> 💬 **What is a for loop?** It tells Python: "repeat this action for every item in a list". Here we loop over every SMILES in the dataset, convert it to a molecule, calculate its properties, and save the results.

Let's now **filter** to only keep drugs that satisfy all Lipinski criteria:


In [ ]:
# ── Keep only drugs that satisfy all four Lipinski criteria ───────────────────
lipinski_filter = (
    (FDA['MolWt'] <= 500) &
    (FDA['HBD']   <= 5)   &
    (FDA['HBA']   <= 10)  &
    (FDA['TPSA']  <  140)
)

FDA_lipinski = FDA[lipinski_filter]

print(f"Total drugs in dataset        : {len(FDA)}")
print(f"Drugs passing Lipinski filter : {len(FDA_lipinski)}")
print(f"Fraction passing              : {len(FDA_lipinski)/len(FDA)*100:.1f}%")
FDA_lipinski.head()

> 💬 How many drugs passed the filter? What fraction of all FDA drugs satisfy Lipinski's rules?  
> (Remember: the Rule of Five is a guideline, not an absolute rule — some drugs intentionally break it.)

---


## Section 6 — Measuring Molecular Similarity 🔍

### Why do we care about similarity?

If we know a molecule is a good drug, we want to find **similar molecules** that might be even better (more effective, fewer side effects, easier to manufacture). This is called **virtual screening**.

But how do you measure how "similar" two molecules are?

### Molecular Fingerprints

The trick is to convert each molecule into a **fingerprint** — a long sequence of 0s and 1s, where each bit represents whether a particular chemical fragment is present or absent in the molecule.


Another very useful task that can be performed with RDkit is to quantify the **similarity between two compounds**. This is important for example when screening libraries for compounds that are similar to known active molecules/drugs.

Similarity can be calculated in different ways: the **Tanimoto coefficient** is one of the most common.

![Tanimoto](tanimoto.png)

In the (simplified!) procedure shown above, each molecule is first encoded into a sequence of binary digits according to the presence or absence of fragments. This sequence is called the *fingerprint* of the molecule. The Tanimoto coefficient is then calculated using the fingerprints of the two molecules as shown above. It's useful to remember that this coefficient can go from 0 (no similarity) to 1 (identity).

### 6a — Introducing imatinib (Gleevec)

**Imatinib** is a landmark drug: it was the first targeted cancer therapy, approved in 2001 for chronic myeloid leukaemia (CML). It works by blocking the BCR-ABL tyrosine kinase — an abnormal protein present only in cancer cells.

Let's draw it:


In [ ]:
imatinib = Chem.MolFromSmiles("CC1=C(C=C(C=C1)NC(=O)C2=CC=C(C=C2)CN3CCN(CC3)C)NC4=NC=CC(=N4)C5=CN=CC=C5")
Draw.MolToImage(imatinib, size = (400, 400))

> 💬 Imatinib is a large, complex molecule — can you identify any of the ring systems?  
> Count how many ring-closure numbers appear in its SMILES.

---
### 6b — Calculating the Tanimoto similarity

Let's compare imatinib to the first compound in our FDA dataset:


In [ ]:
mol2 = Chem.MolFromSmiles(FDA.loc[FDA.index[0], 'SMILES'])
Draw.MolToImage(mol2, size = (200, 200))

Now we compute their **fingerprints** and calculate the **Tanimoto coefficient**:


In [ ]:
fp1 = Chem.RDKFingerprint(imatinib)
fp2 = Chem.RDKFingerprint(mol2)

In [ ]:
DataStructs.FingerprintSimilarity(fp1, fp2)

> 💬 The value is close to 0 — meaning these two compounds are very different. A value of **1.0** would mean identical molecules.

---
### 6c — Finding the most similar drug to imatinib in the FDA database

Now the exciting part — let's scan the **entire FDA database** to find which approved drug is most structurally similar to imatinib:


In [ ]:
MySimilarity = []
for smiles in FDA['SMILES']:
    mol = Chem.MolFromSmiles(smiles)
    fp = Chem.RDKFingerprint(mol)
    MySimilarity.append(DataStructs.FingerprintSimilarity(fp1, fp))
MySimilarity

In [ ]:
FDA['Similarity_to_Imatinib'] = MySimilarity # add MySimilarity to FDA (as last column) and name the new column 'Similarity to Imatinib'

In [ ]:
FDA.sort_values(by = 'Similarity_to_Imatinib', ascending = False) # sort FDA by 'Similarity_to_Imatinib' values

> 💬 Which compound came out on top? Note its ZINC ID — we'll look it up next.


In [ ]:
Draw.MolToImage(imatinib, size = (300, 300)) # imatinib

In [ ]:
mol_best = Chem.MolFromSmiles(FDA.loc['ZINC000006716957', 'SMILES'])
Draw.MolToImage(mol_best, size = (200, 200)) # best match

> 💬 **The answer:** The most similar compound is **nilotinib** — another anti-cancer drug also used for CML, developed specifically to overcome resistance to imatinib. The algorithm found a clinically meaningful relationship without any prior knowledge of biology!

> This is the essence of **computational drug discovery**: using molecular similarity to navigate chemical space and identify promising candidates.

---


## Section 7 — Discussion & Further Thinking 💬

### Questions to discuss with your group:

1. **SMILES and AI:** Large language models like ChatGPT have been trained on text. Researchers are now training similar models on SMILES strings to design new drug molecules. Does this make sense to you? What might the model "learn"?

2. **Lipinski's Rule of Five:** Some modern drugs (like antibiotics and biologics) violate Lipinski's rules entirely. Does this mean the rules are wrong, or just incomplete?

3. **Tanimoto similarity:** Our similarity search found nilotinib — a drug explicitly designed to be similar to imatinib. But what if we searched for compounds with *low* similarity to all known drugs? What might we find?

4. **Scale:** The FDA dataset has a few hundred drugs. Real pharmaceutical databases contain **billions** of compounds. How would that change the computational challenge?

---

### What comes next?

| Topic | What it is |
|-------|-----------|
| **QSAR modelling** | Predicting biological activity from molecular descriptors using ML |
| **Graph Neural Networks** | Treating molecules as graphs and learning directly from their structure |
| **Molecular docking** | Simulating how a drug binds to a protein target in 3D |
| **Generative AI for molecules** | Using AI to *design* new molecules with desired properties |

---

### Further Reading

- Weininger's SMILES story: https://www.chemistryworld.com/opinion/weiningers-smiles/4014639.article
- SMILES theory (full spec): https://www.daylight.com/dayhtml/doc/theory/theory.smiles.html
- RDKit documentation: https://www.rdkit.org/docs/GettingStartedInPython.html
- RDKit blog: https://greglandrum.github.io/rdkit-blog/
- Original SMILES paper: Weininger, *J. Chem. Inf. Comput. Sci.*, 1988, **28**, 31 (DOI: 10.1021/ci00057a005)

---
*Notebook developed for the QMUL AI Literacy Project Taster Day · Department of Chemistry*
